In [5]:
from src.utils import models_dir
from src.config import ExperimentConfig, experiment_config
from dataclasses import dataclass
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    PreTrainedTokenizerBase,
)
from peft import PeftModel
import torch

In [2]:
def load_model(experiment_config: ExperimentConfig) -> tuple[PeftModel, PreTrainedTokenizerBase]:
    tokenizer = AutoTokenizer.from_pretrained(experiment_config.model_hf_path)
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    quantization_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type = 'nf4',
        bnb_4bit_compute_dtype = torch.bfloat16
    )

    model = AutoModelForCausalLM.from_pretrained(
        experiment_config.model_hf_path,
        quantization_config = quantization_config,
        device_map = 'auto',
    )

    model.load_adapter(models_dir / 'coraal_merged', adapter_name='coraal_merged')
    model.load_adapter(models_dir / 'aave_lyrics', adapter_name='aave_lyrics')
    model.load_adapter(models_dir / 'twitter_aae', adapter_name='twitter_aae')
    model.disable_adapters()

    return model, tokenizer

def generate_text(
    prompt: str,
    adapter_name: str,
    model: PeftModel,
    tokenizer: PreTrainedTokenizerBase,
    max_new_tokens: int=50,
    temperature: float=0.7
    ) -> str:
    input_toks = tokenizer(prompt, return_tensors='pt').to('cuda')

    if adapter_name == 'none':
        output_toks_tensor = model.generate(
            **input_toks,
            max_new_tokens = max_new_tokens,
            do_sample = True,
            temperature = temperature,
        )
    else:
        model.enable_adapters()
        model.set_adapter(adapter_name)
        output_toks_tensor = model.generate(
            **input_toks,
            max_new_tokens = max_new_tokens,
            do_sample = True,
            temperature = temperature,
        )
        model.disable_adapters()
    output_toks = output_toks_tensor[0]
    output_text = tokenizer.decode(output_toks, skip_special_tokens=True)

    return output_text

In [3]:
model, tokenizer = load_model(experiment_config)

In [4]:
prompt = "i"
adapter_name = 'aave_lyrics'

generate_text(prompt, adapter_name, model, tokenizer)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


'i i dont care about my ass'